In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing import Annotated
from langgraph.checkpoint.memory import MemorySaver

In [17]:
class ChatState(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]

load_dotenv()
model = ChatOpenAI()

In [18]:
def chat_node(state: ChatState) :
    messages = state['message']
    response = model.invoke(messages)

    return {'message': [response]}


In [21]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

workflow = graph.compile(checkpointer=checkpointer)


In [23]:
thread_id = 1

while True:
    user_input = input('You: ')
    if user_input.strip().lower() in ['exit', 'quit']:
        break
    
    initial_state = {'message':HumanMessage(content=user_input)}

    config = {'configurable': {'thread_id': thread_id}}
    response = workflow.invoke(initial_state, config=config)['message'][-1].content
    print(f'Bot: {response}')

Bot: Hello Spartacus, nice to meet you! How can I assist you today?
Bot: Your name is Spartacus.
Bot: I'm actually a virtual assistant, but I'm glad you think I'm doing a good job! How can I help you today, Spartacus?
